# Exercise 17.1: Simulating the FitzHugh-Nagumo model

We will now upgrade our bistable FDM solver to solve the full FHN model. We add the new parameters for the recovery variable: $\epsilon = 0.005$ and $\gamma = 2.0$.


## Exercise 17.1a: Adding the recovery variable

Extend your vectorized solver from the previous chapter to include the $w$ variable. You will need to calculate the reaction term for both $V$ and $w$ at every time step.

_(Note: $w$ does not diffuse physically through space, so there is no spatial second derivative term for $w$!)_


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython import display
import time

# Parameters
k = 2.0
A = 1.0
alpha = 0.1
L = 100
eps = 0.005
gamma = 2.0

dx = 1
dt = 0.1
N = int(L / dx)

# Initialize arrays
v = np.zeros(N + 1)
w = np.zeros(N + 1)

# Apply stimulus to the left edge
left = int(N / 10)
v[:left] = 0.3

# Slicing arrays for internal nodes
I = np.arange(1, N)
Ip = I + 1
Im = I - 1

for i in range(1400):
    # Safely copy previous states
    v_prev = np.copy(v)
    w_prev = np.copy(w)

    # 1. Calculate reaction terms
    I_ion_v = ...
    I_ion_w = ...

    # 2. Add diffusion to v
    v[I] = ...

    # 3. Update boundary nodes for v
    v[0] = ...
    v[N] = ...

    # 4. Add reaction terms to update v and w
    v = ...
    w = ...

    # Live Plotting
    if i % 20 == 0:
        plt.clf()
        plt.axis([0, L, -0.2, 1.1])
        plt.plot(v, color="C0", linewidth=2, label="Voltage (V)")
        plt.plot(w, color="C1", linewidth=2, linestyle="--", label="Recovery (w)")
        plt.title(f"FHN Model - Time step i={i}")
        plt.legend(loc="upper right")
        display.clear_output(wait=True)
        display.display(plt.gcf())
        time.sleep(0.01)

## Exercise 17.1b: Periodic boundary conditions

Instead of a straight wire with sealed ends, we want to simulate a ring of tissue (like the circumference of a heart chamber). We can do this mathematically by implementing **periodic boundary conditions**.

This simply means that the "left neighbor" of node $0$ is node $N$, and the "right neighbor" of node $N$ is node $0$!

In NumPy, we can achieve this beautifully without any `if` statements by making `I` cover the _entire_ array, and manually wrapping the ends of our neighbor slice arrays (`Ip` and `Im`). Modify the code below to see the wave travel off the right edge of the screen and instantly reappear on the left!


In [ ]:
# Reset arrays
v = np.zeros(N + 1)
w = np.zeros(N + 1)

# Stimulus in the middle of the cable
mid = int(N / 2)
v[mid - 10 : mid + 10] = 0.3

# Wrap-around slicing arrays!
I = np.arange(N + 1)
Ip = I + 1
Ip[N] = 0  # The right neighbor of the last node is the first node!
Im = I - 1
Im[0] = N  # The left neighbor of the first node is the last node!

for i in range(1400):
    v_prev = np.copy(v)
    w_prev = np.copy(w)

    I_ion_v = A * v_prev * (1 - v_prev) * (v_prev - alpha) - w_prev
    I_ion_w = eps * (v_prev - gamma * w_prev)

    # Apply diffusion to the ENTIRE array at once using our wrapped slices!
    v[I] = v_prev[I] + dt * (k / dx**2) * (v_prev[Ip] - 2 * v_prev[I] + v_prev[Im])

    # Add reaction terms
    v = v + dt * I_ion_v
    w = w + dt * I_ion_w

    # Live Plotting
    if i % 20 == 0:
        plt.clf()
        plt.axis([0, L, -0.2, 1.1])
        plt.plot(v, color="C3", linewidth=2)
        plt.title(f"Periodic Boundaries - Time step i={i}")
        display.clear_output(wait=True)
        display.display(plt.gcf())
        time.sleep(0.01)

_Question:_ When you stimulate the middle of the ring, what happens when the two wave fronts eventually collide on the opposite side of the ring? Why?


## Exercise 17.1c: Simulating cardiac reentry

A **reentry circuit** occurs when an electrical wave gets trapped traveling in a continuous loop, never stopping. This is the mechanism behind ventricular fibrillation, the most deadly cardiac arrhythmia.

Normally, waves cannot travel backward because the tissue they just passed through is _refractory_ (the $w$ variable is still high, preventing immediate reactivation). Therefore, when two waves collide, they annihilate each other.

To create a reentry circuit, the wave must only travel in **one direction**.

We can force this by creating an artificial "block" on one side of our initial stimulus. We will artificially inject a high $w$ value (refractory tissue) immediately to the left of our stimulus. Copy your periodic boundary code from above, but change the initial conditions to the following:

```python
mid = int(N / 2)
v[mid-10:mid+10] = 0.3      # Stimulate the center
w[:mid-5] = 0.2             # Make the left side completely refractory!
```

Increase the loop range to `14000` steps and run the simulation!


In [ ]:
# Your reentry simulation code here